# 05. Evaluación final y productivización

Este notebook realiza una única evaluación sobre 2025 después de congelar todas las decisiones con la información disponible hasta 2024.

El proceso persigue cuatro objetivos.

* Confirmar el rendimiento fuera del periodo de desarrollo
* Medir la incertidumbre de las métricas principales
* Auditar el comportamiento por sexo y edad
* Guardar un modelo que pueda recibir observaciones nuevas y devolver predicciones

La primera comprobación impide continuar mientras la configuración del notebook 04 no esté confirmada.

In [ ]:
from importlib.metadata import version
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    confusion_matrix,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tfm_accidentes.config import PROCESSED_DATA_DIR
from tfm_accidentes.data import read_analysis_table
from tfm_accidentes.model_selection import binary_classification_metrics
from tfm_accidentes.production import (
    build_final_estimator,
    create_artifact,
    load_artifact,
    predict_accident_severity,
    required_input_columns,
    save_artifact,
    validate_final_configuration,
)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)

## 1. Configuración congelada

Copia en la celda siguiente la decisión definitiva obtenida en el notebook 04.

Las familias individuales admitidas son `logistic_basic`, `logistic_advanced`, `linear_svm`, `rbf_svm`, `random_forest`, `hist_gradient_boosting` y `xgboost`.

Los parámetros deben coincidir exactamente con los seleccionados en 2024. El umbral también debe copiarse sin volver a optimizarlo.

Mantén `confirmed` como `False` hasta haber comprobado todos los valores. El notebook se detendrá antes de cargar los datos de 2025 si la configuración está incompleta.

In [ ]:
FINAL_CONFIGURATION = {
    'confirmed': False,
    'source_notebook': '04_modelizacion_avanzada.ipynb',
    'model': {
        'kind': 'single',
        'family': 'logistic_advanced',
        'parameters': {},
        'include_sex': True,
    },
    'threshold': None,
    'minimum_recall': 0.75,
    'validation_2024_metrics': None,
}

# Ejemplo de parámetros para una familia de árboles
# FINAL_CONFIGURATION['model']['parameters'] = {
#     'n_estimators': 300,
#     'max_depth': 20,
# }

# Ejemplo de configuración para un blend
# FINAL_CONFIGURATION['model'] = {
#     'kind': 'blend',
#     'first_weight': 0.65,
#     'first_model': {
#         'kind': 'single',
#         'family': 'hist_gradient_boosting',
#         'parameters': {},
#         'include_sex': True,
#     },
#     'second_model': {
#         'kind': 'single',
#         'family': 'logistic_advanced',
#         'parameters': {},
#         'include_sex': True,
#     },
# }

In [ ]:
validate_final_configuration(FINAL_CONFIGURATION)

print('Configuración final confirmada')
print(f"Modelo: {FINAL_CONFIGURATION['model']}")
print(f"Umbral congelado: {FINAL_CONFIGURATION['threshold']:.6f}")

## 2. Preparación de la evaluación

El modelo final se entrena con los registros comprendidos entre 2019 y 2024.

El año 2025 se conserva como test final. Sus etiquetas no intervienen en la selección del modelo, los parámetros, las variables ni el umbral.

In [ ]:
data_path = PROCESSED_DATA_DIR / 'accidentes_madrid_2019_2025.csv.gz'
data = read_analysis_table(data_path)
data = data.loc[data['lesion_grave'].notna()].copy()
data['lesion_grave'] = data['lesion_grave'].astype('int8')

training_data = data.loc[data['anio'].between(2019, 2024)].copy()
sealed_test = data.loc[data['anio'].eq(2025)].copy()

assert training_data['anio'].max() < sealed_test['anio'].min()
assert set(training_data['anio'].unique()) == {2019, 2020, 2021, 2022, 2023, 2024}
assert set(sealed_test['anio'].unique()) == {2025}

required_columns = required_input_columns(FINAL_CONFIGURATION['model'])
X_train_final = training_data[required_columns]
y_train_final = training_data['lesion_grave']
X_test_final = sealed_test[required_columns]

split_summary = pd.DataFrame({
    'periodo': ['Entrenamiento final', 'Test final'],
    'años': ['2019 a 2024', '2025'],
    'filas': [len(training_data), len(sealed_test)],
})
display(split_summary)

## 3. Entrenamiento definitivo

El estimador se construye con la configuración congelada y se ajusta una sola vez con todos los datos disponibles hasta el final de 2024.

Las transformaciones forman parte del mismo pipeline. Esto permite aplicar exactamente la misma preparación cuando lleguen observaciones nuevas.

In [ ]:
final_estimator = build_final_estimator(FINAL_CONFIGURATION['model'])

print('Iniciando el entrenamiento definitivo', flush=True)
fit_start = time.perf_counter()
final_estimator.fit(X_train_final, y_train_final)
final_fit_time = time.perf_counter() - fit_start

print(f'Entrenamiento completado en {final_fit_time:.1f} segundos', flush=True)

## 4. Evaluación única sobre 2025

Esta celda abre las etiquetas del test final. Las probabilidades y las decisiones se calculan con el modelo y el umbral ya congelados.

No deben modificarse decisiones anteriores después de observar estos resultados.

In [ ]:
if globals().get('FINAL_TEST_EXECUTED', False):
    raise RuntimeError('El test final ya se evaluó en este kernel')

FINAL_TEST_EXECUTED = True
y_test_final = sealed_test['lesion_grave']
positive_position = int(np.flatnonzero(final_estimator.classes_ == 1)[0])
test_probabilities = final_estimator.predict_proba(X_test_final)[:, positive_position]
frozen_threshold = float(FINAL_CONFIGURATION['threshold'])
test_predictions = (test_probabilities >= frozen_threshold).astype('int8')

final_metrics = binary_classification_metrics(
    y_test_final,
    test_probabilities,
    threshold=frozen_threshold,
)

final_results = pd.DataFrame([{
    'periodo': 'Test final 2025',
    'prevalencia': y_test_final.mean(),
    'umbral': frozen_threshold,
    'tiempo_entrenamiento_s': final_fit_time,
    **final_metrics,
}]).set_index('periodo')

display(final_results.round(4))

In [ ]:
matrix = confusion_matrix(y_test_final, test_predictions)
tn, fp, fn, tp = matrix.ravel()

operational_summary = pd.DataFrame({
    'concepto': [
        'Casos evaluados',
        'Lesiones graves reales',
        'Casos señalados',
        'Lesiones graves detectadas',
        'Falsas alarmas',
        'Lesiones graves no detectadas',
    ],
    'valor': [len(y_test_final), int(y_test_final.sum()), int(test_predictions.sum()), tp, fp, fn],
})
display(operational_summary)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_final,
    test_predictions,
    display_labels=['No grave', 'Grave'],
    cmap='Blues',
    ax=ax,
)
ax.set_title('Matriz de confusión en 2025')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

PrecisionRecallDisplay.from_predictions(
    y_test_final,
    test_probabilities,
    name='Modelo final',
    ax=axes[0],
)
axes[0].axhline(y_test_final.mean(), color='grey', linestyle=':')
axes[0].set_title('Curva de precisión y exhaustividad')

RocCurveDisplay.from_predictions(
    y_test_final,
    test_probabilities,
    name='Modelo final',
    ax=axes[1],
)
axes[1].plot([0, 1], [0, 1], color='grey', linestyle=':')
axes[1].set_title('Curva ROC')

CalibrationDisplay.from_predictions(
    y_test_final,
    test_probabilities,
    n_bins=10,
    strategy='quantile',
    name='Modelo final',
    ax=axes[2],
)
axes[2].set_title('Calibración probabilística')

plt.tight_layout()
plt.show()

## 5. Incertidumbre de las métricas

Las métricas puntuales dependen de la muestra observada en 2025. El remuestreo estratificado conserva el número de positivos y negativos y permite estimar intervalos del 95 por ciento.

El mismo umbral congelado se aplica en todas las réplicas.

In [ ]:
def stratified_bootstrap_intervals(
    y_true,
    probabilities,
    threshold,
    n_resamples=500,
    confidence_level=0.95,
    random_state=42,
):
    y_array = np.asarray(y_true)
    probability_array = np.asarray(probabilities)
    positive_indices = np.flatnonzero(y_array == 1)
    negative_indices = np.flatnonzero(y_array == 0)
    rng = np.random.default_rng(random_state)
    metric_names = ['pr_auc', 'roc_auc', 'brier', 'precision', 'recall', 'f2']
    samples = {name: np.empty(n_resamples) for name in metric_names}

    for position in range(n_resamples):
        sample_indices = np.concatenate([
            rng.choice(positive_indices, len(positive_indices), replace=True),
            rng.choice(negative_indices, len(negative_indices), replace=True),
        ])
        metrics = binary_classification_metrics(
            y_array[sample_indices],
            probability_array[sample_indices],
            threshold=threshold,
        )
        for name in metric_names:
            samples[name][position] = metrics[name]

    alpha = 1 - confidence_level
    rows = []
    for name in metric_names:
        rows.append({
            'metrica': name,
            'estimacion': final_metrics[name],
            'limite_inferior': np.quantile(samples[name], alpha / 2),
            'limite_superior': np.quantile(samples[name], 1 - alpha / 2),
        })
    return pd.DataFrame(rows)


bootstrap_intervals = stratified_bootstrap_intervals(
    y_test_final,
    test_probabilities,
    frozen_threshold,
    n_resamples=500,
    random_state=RANDOM_STATE,
)
display(bootstrap_intervals.round(4))

## 6. Estabilidad respecto a 2024

Si se copian las métricas externas de 2024 en la configuración inicial, esta sección cuantifica el cambio observado en 2025.

La comparación describe estabilidad temporal. No se utiliza para modificar el modelo ni el umbral.

In [ ]:
reference_2024 = FINAL_CONFIGURATION.get('validation_2024_metrics')

if reference_2024 is None:
    print('No se proporcionaron métricas de referencia de 2024')
else:
    comparison_rows = []
    for metric_name in ['pr_auc', 'roc_auc', 'brier', 'precision', 'recall', 'f2']:
        value_2024 = float(reference_2024[metric_name])
        value_2025 = float(final_metrics[metric_name])
        comparison_rows.append({
            'metrica': metric_name,
            'valor_2024': value_2024,
            'valor_2025': value_2025,
            'cambio_absoluto': value_2025 - value_2024,
        })
    temporal_comparison = pd.DataFrame(comparison_rows)
    display(temporal_comparison.round(4))

## 7. Auditoría por grupos

Se utiliza el mismo umbral global para todos los grupos. Esta decisión permite observar diferencias reales en precisión y exhaustividad.

Los grupos con pocos casos positivos se excluyen de la tabla porque sus métricas serían muy inestables.

In [ ]:
def subgroup_audit(group_column, minimum_positives=15):
    group_values = sealed_test[group_column].fillna('No consta').astype(str)
    rows = []

    for group_value in sorted(group_values.unique()):
        mask = group_values.eq(group_value).to_numpy()
        positives = int(y_test_final.to_numpy()[mask].sum())
        if positives < minimum_positives:
            continue

        metrics = binary_classification_metrics(
            y_test_final.to_numpy()[mask],
            test_probabilities[mask],
            threshold=frozen_threshold,
        )
        rows.append({
            'grupo': group_value,
            'n': int(mask.sum()),
            'positivos': positives,
            'prevalencia': float(y_test_final.to_numpy()[mask].mean()),
            **metrics,
        })

    return pd.DataFrame(rows).sort_values('recall')


audit_sex = subgroup_audit('sexo')
audit_age = subgroup_audit('rango_edad')

display(audit_sex.round(4))
display(audit_age.round(4))

## 8. Creación del artefacto productivo

El artefacto contiene el pipeline entrenado, el umbral, las columnas requeridas, la configuración y las versiones principales de las librerías.

El archivo permite predecir observaciones nuevas sin ejecutar de nuevo los notebooks de análisis y entrenamiento.

In [ ]:
library_versions = {
    'python': sys.version.split()[0],
    'pandas': version('pandas'),
    'numpy': version('numpy'),
    'scikit_learn': version('scikit-learn'),
    'joblib': version('joblib'),
}

try:
    library_versions['xgboost'] = version('xgboost')
except Exception:
    library_versions['xgboost'] = 'No instalado'

artifact = create_artifact(
    final_estimator,
    FINAL_CONFIGURATION,
    training_rows=len(training_data),
    library_versions=library_versions,
)

artifact_path = PROJECT_ROOT / 'models' / 'modelo_final.joblib'
save_artifact(artifact, artifact_path)

print(f'Artefacto guardado en {artifact_path}')

In [ ]:
loaded_artifact = load_artifact(artifact_path)
verification_sample = X_test_final.head(100)

predictions_before = predict_accident_severity(verification_sample, artifact)
predictions_after = predict_accident_severity(verification_sample, loaded_artifact)

np.testing.assert_allclose(
    predictions_before['probabilidad_lesion_grave'],
    predictions_after['probabilidad_lesion_grave'],
)
np.testing.assert_array_equal(
    predictions_before['prediccion_lesion_grave'],
    predictions_after['prediccion_lesion_grave'],
)

print('La serialización conserva exactamente las predicciones')

## 9. Ejemplo de uso

La función productiva recibe un DataFrame con una o varias personas implicadas.

Antes de predecir comprueba que existan todas las columnas necesarias y que las variables temporales estén dentro de sus intervalos esperados.

El resultado contiene la probabilidad estimada y la decisión obtenida con el umbral congelado.

In [ ]:
new_observations = X_test_final.head(3).copy()
prediction_example = predict_accident_severity(
    new_observations,
    loaded_artifact,
)

display(prediction_example.round(4))

## 10. Lectura final

La evaluación debe interpretarse atendiendo a la baja prevalencia y al coste de los errores.

* La PR AUC resume la calidad de la ordenación de los casos graves
* La precisión indica cuántas alertas corresponden realmente a lesiones graves
* La exhaustividad indica qué proporción de lesiones graves se detecta
* El Brier evalúa la calidad de las probabilidades
* La auditoría por grupos permite identificar diferencias de comportamiento

El resultado de 2025 no se utiliza para reajustar el sistema. Cualquier mejora posterior debe considerarse una nueva versión y evaluarse con un periodo futuro diferente.

In [ ]:
print('Resumen final')
print(f"Modelo: {FINAL_CONFIGURATION['model']}")
print(f'Umbral congelado: {frozen_threshold:.6f}')
print(f"PR AUC: {final_metrics['pr_auc']:.4f}")
print(f"ROC AUC: {final_metrics['roc_auc']:.4f}")
print(f"Brier: {final_metrics['brier']:.4f}")
print(f"Precisión: {final_metrics['precision']:.4f}")
print(f"Exhaustividad: {final_metrics['recall']:.4f}")
print(f"F2: {final_metrics['f2']:.4f}")
print(f'Casos señalados: {int(test_predictions.sum())}')
print(f'Lesiones graves no detectadas: {fn}')